# AU Tokens50 — Hash + SimHash Builder with Small Batches and Real-Time Batch Output

This notebook processes the AU parquet files under:

```text
/home/jovyan/data/AUTokens50
```

For each original parquet file, it creates one corresponding output parquet file:

```text
/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_0.parquet
...
/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_56.parquet
```

## Teacher requirement

Add two new columns:

```text
hash
simhash
```

Rules implemented:

1. Do not remove existing columns.
2. Do not overwrite existing columns.
3. Add `hash` and `simhash` as new columns only.
4. Build both only from the actual text column: `text`.
5. Do not use metadata, filename, source field, token count, URL, path, row index, or any other column.

## Upgrade in this version

This version is designed for large parquet files such as `part_0.parquet` around 1.8 GB.

Main changes:

1. **Smaller default batch size**
   - `BATCH_SIZE = 10_000`
   - This makes the notebook show progress much sooner.

2. **Real-time output after every batch**
   - After each batch finishes, the notebook prints:
     - parquet file name
     - batch number
     - rows processed
     - total rows
     - batch time
     - total elapsed time

3. **Real-time output after every parquet**
   - After each parquet completes, it prints:
     - status
     - rows written
     - number of batches
     - elapsed time
     - output path

4. **Skip completed outputs**
   - If a parquet was already processed successfully, rerunning the notebook skips it.

5. **Optional single-file test mode**
   - You can run only `part_0.parquet` first to confirm speed and progress display.

## Cell 0 — Optional package installation

**Purpose**

Install required packages if the current Jupyter kernel does not already have them.

**Method**

Use `sys.executable` so packages are installed into the same Python environment used by the notebook.

**When to run**

Run this cell only if `import pyarrow`, `import pandas`, or `import tqdm` fails.

In [1]:
# Run only if imports fail.

import sys
!{sys.executable} -m pip install -U pyarrow pandas tqdm

In [2]:
from pathlib import Path
import json
import time
import hashlib
import os
from concurrent.futures import ProcessPoolExecutor, as_completed

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

# -----------------------------
# Paths
# -----------------------------
CANDIDATE_DATA_ROOTS = [
    Path("data/AUTokens50"),
    Path("../data/AUTokens50"),
    Path("/home/jovyan/data/AUTokens50"),
    Path("/home/jovyan/notebook/data/AUTokens50"),
]

DATA_ROOT = None
for candidate in CANDIDATE_DATA_ROOTS:
    if candidate.exists():
        DATA_ROOT = candidate.resolve()
        break

if DATA_ROOT is None:
    DATA_ROOT = Path("/home/jovyan/data/AUTokens50").resolve()

OUTPUT_ROOT = Path("/home/jovyan/notebook/outputs/AUTokens50_hash_simhash")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

STATS_FILE = OUTPUT_ROOT / "hash_simhash_stats.json"

# -----------------------------
# Required columns
# -----------------------------
TEXT_COL = "text"
HASH_COL = "hash"
SIMHASH_COL = "simhash"

# -----------------------------
# Runtime options
# -----------------------------
# Small batch for real-time output and lower memory pressure.
BATCH_SIZE = 50_000

# zstd gives a good speed/size trade-off.
COMPRESSION = "zstd"

# Resume support: skip output files that already exist and pass schema/row checks.
SKIP_COMPLETED = True

# Default: run sequentially for stability on shared JupyterHub.
USE_MULTIPROCESSING = True
NUM_WORKERS = 4

# Test mode: set to True to process only the first parquet file.
RUN_ONLY_FIRST_FILE = False

print("Current working directory:", Path.cwd())
print("DATA_ROOT:", DATA_ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("BATCH_SIZE:", BATCH_SIZE)
print("COMPRESSION:", COMPRESSION)
print("SKIP_COMPLETED:", SKIP_COMPLETED)
print("USE_MULTIPROCESSING:", USE_MULTIPROCESSING)
print("NUM_WORKERS:", NUM_WORKERS)
print("RUN_ONLY_FIRST_FILE:", RUN_ONLY_FIRST_FILE)

Current working directory: /home/jovyan/notebook
DATA_ROOT: /home/jovyan/data/AUTokens50
DATA_ROOT exists: True
OUTPUT_ROOT: /home/jovyan/notebook/outputs/AUTokens50_hash_simhash
BATCH_SIZE: 50000
COMPRESSION: zstd
SKIP_COMPLETED: True
USE_MULTIPROCESSING: True
NUM_WORKERS: 4
RUN_ONLY_FIRST_FILE: False


/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Cell 2 — Locate AU input parquet files

**Purpose**

Find all source parquet files to process.

**Method**

The notebook scans only files named:

```text
part_*.parquet
```

and sorts them numerically.

**Why this matters**

The AU directory also contains `total.json` and `total.txt`. They are not parquet data files and should not be processed.

In [3]:
def part_number(path: Path) -> int:
    try:
        return int(path.stem.split("_")[-1])
    except Exception:
        return 10**18

input_files = sorted(DATA_ROOT.glob("part_*.parquet"), key=part_number)

print(f"Found {len(input_files)} input parquet files.")

if len(input_files) == 0:
    print("\nNo part_*.parquet files found. Check candidate paths:")
    for p in CANDIDATE_DATA_ROOTS:
        print(f"- {p.resolve()} | exists={p.exists()}")

assert len(input_files) > 0, f"No part_*.parquet files found under {DATA_ROOT}"

print("\nFirst files:")
for p in input_files[:10]:
    print("-", p)

print("\nLast files:")
for p in input_files[-5:]:
    print("-", p)

if RUN_ONLY_FIRST_FILE:
    input_files_to_process = input_files[:1]
else:
    input_files_to_process = input_files

print("\nFiles selected for processing:", len(input_files_to_process))

Found 57 input parquet files.

First files:
- /home/jovyan/data/AUTokens50/part_0.parquet
- /home/jovyan/data/AUTokens50/part_1.parquet
- /home/jovyan/data/AUTokens50/part_2.parquet
- /home/jovyan/data/AUTokens50/part_3.parquet
- /home/jovyan/data/AUTokens50/part_4.parquet
- /home/jovyan/data/AUTokens50/part_5.parquet
- /home/jovyan/data/AUTokens50/part_6.parquet
- /home/jovyan/data/AUTokens50/part_7.parquet
- /home/jovyan/data/AUTokens50/part_8.parquet
- /home/jovyan/data/AUTokens50/part_9.parquet

Last files:
- /home/jovyan/data/AUTokens50/part_52.parquet
- /home/jovyan/data/AUTokens50/part_53.parquet
- /home/jovyan/data/AUTokens50/part_54.parquet
- /home/jovyan/data/AUTokens50/part_55.parquet
- /home/jovyan/data/AUTokens50/part_56.parquet

Files selected for processing: 57


## Cell 3 — Inspect schemas and protect existing columns

**Purpose**

Check the source parquet schemas before processing.

**Method**

For each source parquet, read only metadata/schema and verify:

1. `text` exists.
2. `hash` does not already exist.
3. `simhash` does not already exist.

**Why this matters**

The teacher requirement says not to overwrite existing columns. If a source parquet already has `hash` or `simhash`, this notebook stops instead of overwriting.

In [4]:
schema_rows = []

for p in tqdm(input_files_to_process, desc="Inspecting source schemas"):
    pf = pq.ParquetFile(p)
    cols = pf.schema.names
    schema_rows.append({
        "file": str(p),
        "rows": pf.metadata.num_rows,
        "size_gb": p.stat().st_size / 1024**3,
        "has_text": TEXT_COL in cols,
        "has_hash": HASH_COL in cols,
        "has_simhash": SIMHASH_COL in cols,
        "num_columns": len(cols),
        "columns": ",".join(cols),
    })

schema_df = pd.DataFrame(schema_rows)
display(schema_df.head())

missing_text = schema_df[~schema_df["has_text"]]
existing_hash = schema_df[schema_df["has_hash"]]
existing_simhash = schema_df[schema_df["has_simhash"]]

print("Files selected:", len(schema_df))
print("Files missing text:", len(missing_text))
print("Files already containing hash:", len(existing_hash))
print("Files already containing simhash:", len(existing_simhash))
print("Total selected rows:", int(schema_df["rows"].sum()))
print("Total selected compressed size GB:", f'{schema_df["size_gb"].sum():.2f}')

assert len(missing_text) == 0, "Some files do not contain the required text column."
assert len(existing_hash) == 0, "Some files already contain a hash column. Refusing to overwrite."
assert len(existing_simhash) == 0, "Some files already contain a simhash column. Refusing to overwrite."

Inspecting source schemas: 100%|██████████| 57/57 [00:00<00:00, 823.53it/s]


,file,rows,size_gb,has_text,has_hash,has_simhash,num_columns,columns
0,/home/jovyan/data/AUTokens50/part_0.parquet,278106,1.772963,True,False,False,10,"text,id,dump,url,date,file_path,language,langu..."
1,/home/jovyan/data/AUTokens50/part_1.parquet,374529,2.381496,True,False,False,10,"text,id,dump,url,date,file_path,language,langu..."
2,/home/jovyan/data/AUTokens50/part_2.parquet,268518,1.612431,True,False,False,10,"text,id,dump,url,date,file_path,language,langu..."
3,/home/jovyan/data/AUTokens50/part_3.parquet,342427,2.105148,True,False,False,10,"text,id,dump,url,date,file_path,language,langu..."
4,/home/jovyan/data/AUTokens50/part_4.parquet,259925,1.972342,True,False,False,10,"text,id,dump,url,date,file_path,language,langu..."


Files selected: 57
Files missing text: 0
Files already containing hash: 0
Files already containing simhash: 0
Total selected rows: 21087435
Total selected compressed size GB: 127.75


## Cell 4 — Define text-only hash and SimHash functions

**Purpose**

Define how `hash` and `simhash` are computed.

**Method**

- `hash`: SHA-1 of the actual text content.
- `simhash`: 64-bit token-level SimHash from the actual text content.

**Why token-level SimHash**

Token-level SimHash is faster than character n-gram SimHash because it processes fewer units for long texts.

**Requirement compliance**

Only `df["text"]` is passed into these functions. The functions do not use filename, path, metadata, URL, source, token count, or any other column.

In [5]:
from collections import Counter

def stable_text_hash(text) -> str:
    """Stable exact hash based only on actual text content."""
    if text is None:
        text = ""
    if not isinstance(text, str):
        text = str(text)

    # Only actual text content is used.
    # No metadata, filename, source field, token_count, URL, path, or row index is used.
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()


def simhash_64_fast(text) -> int:
    """64-bit token SimHash based only on actual text content.

    Faster version:
    - uses token counts to avoid repeatedly processing duplicate tokens
    - uses digest() instead of hexdigest()
    - still uses only the text column
    """
    if text is None:
        text = ""
    if not isinstance(text, str):
        text = str(text)

    tokens = text.lower().split()

    if not tokens:
        return 0

    token_counts = Counter(tokens)
    vector = [0] * 64

    for tok, weight in token_counts.items():
        h = int.from_bytes(
            hashlib.blake2b(
                tok.encode("utf-8", errors="ignore"),
                digest_size=8
            ).digest(),
            byteorder="little",
            signed=False,
        )

        for bit in range(64):
            if h & (1 << bit):
                vector[bit] += weight
            else:
                vector[bit] -= weight

    out = 0
    for bit, score in enumerate(vector):
        if score >= 0:
            out |= (1 << bit)

    return out


def add_text_comparison_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Append hash and simhash using only df[TEXT_COL].

    Existing source columns are preserved.
    Existing hash/simhash columns are not overwritten.
    """
    if TEXT_COL not in df.columns:
        raise ValueError(f"Missing required text column: {TEXT_COL}")

    if HASH_COL in df.columns:
        raise ValueError(f"Column already exists and will not be overwritten: {HASH_COL}")

    if SIMHASH_COL in df.columns:
        raise ValueError(f"Column already exists and will not be overwritten: {SIMHASH_COL}")

    text_series = df[TEXT_COL]

    # Both columns are built only from the actual text column.
    df[HASH_COL] = text_series.map(stable_text_hash)
    df[SIMHASH_COL] = text_series.map(simhash_64_fast).astype("uint64")

    return df

## Cell 5 — Completed-output check for resume support

**Purpose**

Check whether a parquet file was already processed successfully.

**Method**

An output file is considered complete only if:

1. It exists.
2. Its row count equals the source file row count.
3. It contains `text`, `hash`, and `simhash`.

**Why this matters**

If the notebook is interrupted after processing some files, rerunning it will skip completed files and continue from the next incomplete file.

In [6]:
def output_is_complete(input_path: Path, output_path: Path) -> bool:
    if not output_path.exists():
        return False

    try:
        src_pf = pq.ParquetFile(input_path)
        out_pf = pq.ParquetFile(output_path)

        src_rows = src_pf.metadata.num_rows
        out_rows = out_pf.metadata.num_rows
        out_cols = out_pf.schema.names

        return (
            src_rows == out_rows
            and TEXT_COL in out_cols
            and HASH_COL in out_cols
            and SIMHASH_COL in out_cols
        )
    except Exception:
        return False

## Cell 6 — Process one parquet with real-time batch output

**Purpose**

Process one source parquet and write one output parquet.

**Method**

For each batch:

1. Read one batch from the source parquet.
2. Convert the batch to pandas.
3. Preserve all existing source columns.
4. Add `hash` and `simhash` from `text` only.
5. Write the batch to the output parquet.
6. Print real-time batch progress.

**Real-time output**

After every batch, this function prints:

```text
part_0.parquet | batch=1 | rows_done=10,000/...
```

This is the key upgrade for long-running large files.

In [7]:
def process_one_file(input_path: Path, output_path: Path) -> dict:
    start = time.time()

    if SKIP_COMPLETED and output_is_complete(input_path, output_path):
        src_pf = pq.ParquetFile(input_path)
        return {
            "input_file": str(input_path),
            "output_file": str(output_path),
            "rows_written": src_pf.metadata.num_rows,
            "batches_written": 0,
            "elapsed_seconds": 0.0,
            "status": "skipped_existing_complete",
        }

    if output_path.exists():
        output_path.unlink()

    pf = pq.ParquetFile(input_path)
    source_cols = pf.schema.names
    total_rows = pf.metadata.num_rows
    size_gb = input_path.stat().st_size / 1024**3

    if TEXT_COL not in source_cols:
        raise ValueError(f"{input_path} does not contain {TEXT_COL}")

    if HASH_COL in source_cols or SIMHASH_COL in source_cols:
        raise ValueError(f"{input_path} already contains hash/simhash; refusing to overwrite.")

    writer = None
    output_schema = None
    rows_written = 0
    batches_written = 0

    final_cols = list(source_cols) + [HASH_COL, SIMHASH_COL]

    print(
        f"\nSTART {input_path.name} | "
        f"rows={total_rows:,} | "
        f"size={size_gb:.2f}GB | "
        f"batch_size={BATCH_SIZE:,} | "
        f"output={output_path}",
        flush=True
    )

    try:
        for batch_idx, batch in enumerate(pf.iter_batches(batch_size=BATCH_SIZE), start=1):
            batch_start = time.time()

            # Important:
            # ignore_metadata=True prevents a source column named "index"
            # from being converted into the pandas DataFrame index.
            df = batch.to_pandas(ignore_metadata=True)

            # Check that pandas conversion preserved every original source column.
            missing_cols = [c for c in source_cols if c not in df.columns]
            if missing_cols:
                raise ValueError(
                    f"{input_path.name} batch {batch_idx}: "
                    f"missing original columns after pandas conversion: {missing_cols}"
                )

            # Add hash and simhash using only the actual text column.
            df = add_text_comparison_columns(df)

            # Enforce final column order:
            # all original columns first, then hash and simhash.
            df = df[final_cols]

            table = pa.Table.from_pandas(df, preserve_index=False)

            if writer is None:
                output_schema = table.schema
                writer = pq.ParquetWriter(
                    output_path,
                    output_schema,
                    compression=COMPRESSION,
                    use_dictionary=True,
                    write_statistics=True,
                )
            else:
                table = table.cast(output_schema)

            writer.write_table(table)

            rows_written += table.num_rows
            batches_written += 1

            batch_elapsed = time.time() - batch_start
            total_elapsed_so_far = time.time() - start
            rows_per_sec = table.num_rows / batch_elapsed if batch_elapsed > 0 else 0

            print(
                f"  {input_path.name} | "
                f"batch={batch_idx} | "
                f"rows_done={rows_written:,}/{total_rows:,} | "
                f"batch_rows={table.num_rows:,} | "
                f"batch_time={batch_elapsed:.1f}s | "
                f"rows/s={rows_per_sec:,.0f} | "
                f"elapsed={total_elapsed_so_far:.1f}s",
                flush=True
            )

    finally:
        if writer is not None:
            writer.close()

    elapsed = time.time() - start

    print(
        f"DONE {input_path.name} | "
        f"rows={rows_written:,} | "
        f"batches={batches_written} | "
        f"elapsed={elapsed:.1f}s | "
        f"output={output_path}",
        flush=True
    )

    return {
        "input_file": str(input_path),
        "output_file": str(output_path),
        "rows_written": rows_written,
        "batches_written": batches_written,
        "elapsed_seconds": elapsed,
        "status": "processed",
    }

## Cell 8 — Optional multiprocessing mode

**Purpose**

Process multiple parquet files in parallel.

**Method**

`ProcessPoolExecutor` runs several files at once.

**Warning**

For 1.8GB+ parquet files, parallel mode can create heavy memory and disk I/O pressure. Use only if the machine has enough resources. Start with `NUM_WORKERS = 2`.

In [8]:
# Run this cell only if USE_MULTIPROCESSING=True.

if USE_MULTIPROCESSING:
    global_start = time.time()
    all_stats = []

    jobs = []
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as ex:
        for input_path in input_files_to_process:
            output_path = OUTPUT_ROOT / input_path.name
            jobs.append(ex.submit(process_one_file, input_path, output_path))

        for fut in tqdm(as_completed(jobs), total=len(jobs), desc="Processing in parallel"):
            stats = fut.result()
            all_stats.append(stats)

            print(
                f"[{stats['status']}] {Path(stats['input_file']).name} | "
                f"rows={stats['rows_written']:,} | "
                f"batches={stats['batches_written']} | "
                f"elapsed={stats['elapsed_seconds']:.1f}s | "
                f"output={stats['output_file']}",
                flush=True
            )

    total_elapsed = time.time() - global_start

    summary = {
        "dataset": "AUTokens50 with hash and simhash",
        "mode": "multiprocessing",
        "num_workers": NUM_WORKERS,
        "input_dir": str(DATA_ROOT),
        "output_dir": str(OUTPUT_ROOT),
        "num_input_files_total": len(input_files),
        "num_input_files_selected": len(input_files_to_process),
        "num_output_files": len(list(OUTPUT_ROOT.glob("part_*.parquet"))),
        "hash_column": HASH_COL,
        "simhash_column": SIMHASH_COL,
        "hash_source_column": TEXT_COL,
        "simhash_source_column": TEXT_COL,
        "preserve_existing_columns": True,
        "overwrite_existing_columns": False,
        "batch_size": BATCH_SIZE,
        "compression": COMPRESSION,
        "skip_completed": SKIP_COMPLETED,
        "run_only_first_file": RUN_ONLY_FIRST_FILE,
        "elapsed_seconds": total_elapsed,
        "files": all_stats,
    }

    with open(STATS_FILE, "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2, ensure_ascii=False)

    print("\nDONE")
    print("Output directory:", OUTPUT_ROOT)
    print("Stats file:", STATS_FILE)
    print("Total elapsed seconds:", f"{total_elapsed:.1f}")
else:
    print("USE_MULTIPROCESSING=False, so this cell is not needed.")

Processing in parallel:   0%|          | 0/57 [00:00<?, ?it/s]

[skipped_existing_complete] part_0.parquet | rows=278,106 | batches=0 | elapsed=0.0s | output=/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_0.parquet
[skipped_existing_complete] part_2.parquet | rows=268,518 | batches=0 | elapsed=0.0s | output=/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_2.parquet


Processing in parallel:   4%|▎         | 2/57 [00:00<00:02, 19.27it/s]

[skipped_existing_complete] part_4.parquet | rows=259,925 | batches=0 | elapsed=0.0s | output=/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_4.parquet
[skipped_existing_complete] part_3.parquet | rows=342,427 | batches=0 | elapsed=0.0s | output=/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_3.parquet
[skipped_existing_complete] part_1.parquet | rows=374,529 | batches=0 | elapsed=0.0s | output=/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_1.parquet
[skipped_existing_complete] part_5.parquet | rows=288,089 | batches=0 | elapsed=0.0s | output=/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_5.parquet


Processing in parallel:  11%|█         | 6/57 [00:00<00:02, 24.39it/s]

...

[skipped_existing_complete] part_51.parquet | rows=292,760 | batches=0 | elapsed=0.0s | output=/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_51.parquet
[skipped_existing_complete] part_52.parquet | rows=482,084 | batches=0 | elapsed=0.0s | output=/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_52.parquet
[skipped_existing_complete] part_56.parquet | rows=202,508 | batches=0 | elapsed=0.0s | output=/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_56.parquet
[skipped_existing_complete] part_53.parquet | rows=487,144 | batches=0 | elapsed=0.0s | output=/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_53.parquet


Processing in parallel:  96%|█████████▋| 55/57 [00:02<00:00, 20.70it/s]

[skipped_existing_complete] part_54.parquet | rows=486,753 | batches=0 | elapsed=0.0s | output=/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_54.parquet
[skipped_existing_complete] part_55.parquet | rows=488,157 | batches=0 | elapsed=0.0s | output=/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_55.parquet


Processing in parallel: 100%|██████████| 57/57 [00:02<00:00, 24.94it/s]


DONE
Output directory: /home/jovyan/notebook/outputs/AUTokens50_hash_simhash
Stats file: /home/jovyan/notebook/outputs/AUTokens50_hash_simhash/hash_simhash_stats.json
Total elapsed seconds: 2.3


## Cell 9 — Verify output files

**Purpose**

Check that every selected output parquet exists and contains `hash` and `simhash`.

**Method**

Only parquet metadata and schema are read. This is much faster than scanning all text again.

In [9]:
expected_output_files = [OUTPUT_ROOT / p.name for p in input_files_to_process]
existing_output_files = [p for p in expected_output_files if p.exists()]

verify_rows = []

for p in tqdm(existing_output_files, desc="Verifying output schemas"):
    pf = pq.ParquetFile(p)
    cols = pf.schema.names

    verify_rows.append({
        "file": str(p),
        "rows": pf.metadata.num_rows,
        "has_text": TEXT_COL in cols,
        "has_hash": HASH_COL in cols,
        "has_simhash": SIMHASH_COL in cols,
        "num_columns": len(cols),
    })

verify_df = pd.DataFrame(verify_rows)
display(verify_df.head())

print("Expected selected output files:", len(expected_output_files))
print("Existing selected output files:", len(existing_output_files))

if len(verify_df) > 0:
    print("Total output rows:", int(verify_df["rows"].sum()))
    print("Files missing hash:", int((~verify_df["has_hash"]).sum()))
    print("Files missing simhash:", int((~verify_df["has_simhash"]).sum()))

assert len(existing_output_files) == len(expected_output_files), "Output file count does not match selected input file count."
assert verify_df["has_hash"].all(), "Some output files are missing hash."
assert verify_df["has_simhash"].all(), "Some output files are missing simhash."

print("OK: all selected output files contain hash and simhash.")

Verifying output schemas: 100%|██████████| 57/57 [00:00<00:00, 1205.06it/s]


,file,rows,has_text,has_hash,has_simhash,num_columns
0,/home/jovyan/notebook/outputs/AUTokens50_hash_...,278106,True,True,True,12
1,/home/jovyan/notebook/outputs/AUTokens50_hash_...,374529,True,True,True,12
2,/home/jovyan/notebook/outputs/AUTokens50_hash_...,268518,True,True,True,12
3,/home/jovyan/notebook/outputs/AUTokens50_hash_...,342427,True,True,True,12
4,/home/jovyan/notebook/outputs/AUTokens50_hash_...,259925,True,True,True,12


Expected selected output files: 57
Existing selected output files: 57
Total output rows: 21087435
Files missing hash: 0
Files missing simhash: 0
OK: all selected output files contain hash and simhash.


## Cell 10 — Spot-check text-only correctness

**Purpose**

Verify that saved `hash` and `simhash` match recomputation from `text`.

**Method**

Read 10 rows from the first selected output file, recompute from `text`, and compare.

**Why this matters**

This directly checks that the two comparison columns were produced from actual text content.

In [10]:
sample_file = existing_output_files[0]
pf = pq.ParquetFile(sample_file)

sample_batch = next(pf.iter_batches(columns=[TEXT_COL, HASH_COL, SIMHASH_COL], batch_size=10))
sample_df = sample_batch.to_pandas()

sample_df["recomputed_hash"] = sample_df[TEXT_COL].map(stable_text_hash)
sample_df["recomputed_simhash"] = sample_df[TEXT_COL].map(simhash_64_fast).astype("uint64")

sample_df["hash_match"] = sample_df[HASH_COL] == sample_df["recomputed_hash"]
sample_df["simhash_match"] = sample_df[SIMHASH_COL] == sample_df["recomputed_simhash"]

display(sample_df[[HASH_COL, "recomputed_hash", "hash_match", SIMHASH_COL, "recomputed_simhash", "simhash_match"]])

assert sample_df["hash_match"].all(), "Hash verification failed."
assert sample_df["simhash_match"].all(), "SimHash verification failed."

print("OK: sample hash and simhash match recomputation from text only.")

,hash,recomputed_hash,hash_match,simhash,recomputed_simhash,simhash_match
0,fbbcccf9d47475cf0846f80c6581f5fc9adc7a54,fbbcccf9d47475cf0846f80c6581f5fc9adc7a54,True,13205282605077617983,13205282605077617983,True
1,20aafdf7189bd8a55cd4c3635d38c0dcac8c010a,20aafdf7189bd8a55cd4c3635d38c0dcac8c010a,True,4202569391797958750,4202569391797958750,True
2,edf4589dc466903b55dc711cac9dc2006cabc9f8,edf4589dc466903b55dc711cac9dc2006cabc9f8,True,4270193771566332247,4270193771566332247,True
3,657858070683905c9bfc02f159b18c9b4483928a,657858070683905c9bfc02f159b18c9b4483928a,True,4182373561614981694,4182373561614981694,True
4,813c9cc6fea233672d1f508acc6b2fafa1e33da2,813c9cc6fea233672d1f508acc6b2fafa1e33da2,True,3910961314316015894,3910961314316015894,True
5,9b50b3eef62b6bd5c713c892ea68bac589e07251,9b50b3eef62b6bd5c713c892ea68bac589e07251,True,3891816617853188438,3891816617853188438,True
6,33c973284eaff8f2f163077bf85509fb00bd553e,33c973284eaff8f2f163077bf85509fb00bd553e,True,3621481308570245663,3621481308570245663,True
7,bf8e94c9ea8df04d0c35b4782251f6e8d06bce9a,bf8e94c9ea8df04d0c35b4782251f6e8d06bce9a,True,4180051392591419479,4180051392591419479,True
8,ea34df84bfbbda89c3f3aee90b2121131dd16ec1,ea34df84bfbbda89c3f3aee90b2121131dd16ec1,True,4215938902362151007,4215938902362151007,True
9,6a7cda32c27573854530557f40fe02208644a18e,6a7cda32c27573854530557f40fe02208644a18e,True,4180051410840976670,4180051410840976670,True


OK: sample hash and simhash match recomputation from text only.


## Cell 11 — Optional upload command

**Purpose**

Show the upload command for the transformed output folder.

**Note**

Run only after verification passes and after confirming the target Hugging Face repository.

In [11]:
print("Example upload command:")
print()
print("hf upload JoeyLLM/Australia-dataset-50b \\")
print(f"  {OUTPUT_ROOT} \\")
print("  . \\")
print("  --repo-type dataset")

Example upload command:

hf upload JoeyLLM/Australia-dataset-50b \
  /home/jovyan/notebook/outputs/AUTokens50_hash_simhash \
  . \
  --repo-type dataset


CHECK

In [12]:
from pathlib import Path
import pyarrow.parquet as pq

src = Path("/home/jovyan/data/AUTokens50/part_0.parquet")
out = Path("/home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_0.parquet")

src_pf = pq.ParquetFile(src)
out_pf = pq.ParquetFile(out)

print("Source size GB:", src.stat().st_size / 1024**3)
print("Output size GB:", out.stat().st_size / 1024**3)

print("Source rows:", src_pf.metadata.num_rows)
print("Output rows:", out_pf.metadata.num_rows)

print("Source columns:", src_pf.schema.names)
print("Output columns:", out_pf.schema.names)

print("Has hash:", "hash" in out_pf.schema.names)
print("Has simhash:", "simhash" in out_pf.schema.names)

src_batch = next(src_pf.iter_batches(columns=["text"], batch_size=10))
out_batch = next(out_pf.iter_batches(columns=["text", "hash", "simhash"], batch_size=10))

src_df = src_batch.to_pandas()
out_df = out_batch.to_pandas()

print("First 10 text rows unchanged:", (src_df["text"] == out_df["text"]).all())

display(out_df.head())

Source size GB: 1.7729630842804909
Output size GB: 1.2308827191591263
Source rows: 278106
Output rows: 278106
Source columns: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count', 'index']
Output columns: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count', 'index', 'hash', 'simhash']
Has hash: True
Has simhash: True
First 10 text rows unchanged: True


,text,hash,simhash
0,"Justine Davies –, Monday, January, 31, 2011, (...",fbbcccf9d47475cf0846f80c6581f5fc9adc7a54,13205282605077617983
1,Interview: Jenny Macpherson\n- by Rowena Scott...,20aafdf7189bd8a55cd4c3635d38c0dcac8c010a,4202569391797958750
2,The foundations for successful riding\n19 post...,edf4589dc466903b55dc711cac9dc2006cabc9f8,4270193771566332247
3,photos by Carlo Ledesma\nWhen God was creating...,657858070683905c9bfc02f159b18c9b4483928a,4182373561614981694
4,Joanne Harris is apparently as formidable a Yo...,813c9cc6fea233672d1f508acc6b2fafa1e33da2,3910961314316015894


## Manual Verification for a Source–Output Parquet Pair

This cell verifies one selected source parquet file against its corresponding processed output parquet file.

The goal is to confirm that the transformation satisfies the dataset requirement:

- all original columns are preserved;
- no existing column is removed or overwritten;
- `hash` and `simhash` are added as new columns;
- both `hash` and `simhash` are generated only from the actual text column;
- the number of rows remains unchanged;
- sampled text rows remain identical between the source and output files;
- sampled `hash` and `simhash` values can be recomputed from `text` and match the saved values.

You can manually choose which parquet pair to check by changing:

```python
PART_ID = 0

In [14]:
# Cell: Manually verify one source parquet against one output parquet

from pathlib import Path
import pyarrow.parquet as pq
import pandas as pd

# ============================================================
# Manually set which pair to check
# ============================================================

PART_ID = 56         # Change this to check part_1, part_2, ..., part_56
SAMPLE_ROWS = 100    # Number of rows to recompute hash/simhash for

SOURCE_DIR = Path("/home/jovyan/data/AUTokens50")
OUTPUT_DIR = Path("/home/jovyan/notebook/outputs/AUTokens50_hash_simhash")

src = SOURCE_DIR / f"part_{PART_ID}.parquet"
out = OUTPUT_DIR / f"part_{PART_ID}.parquet"

print("Checking parquet pair")
print("Source:", src)
print("Output:", out)

# ============================================================
# Basic existence checks
# ============================================================

assert src.exists(), f"Source parquet does not exist: {src}"
assert out.exists(), f"Output parquet does not exist: {out}"

src_pf = pq.ParquetFile(src)
out_pf = pq.ParquetFile(out)

src_cols = src_pf.schema.names
out_cols = out_pf.schema.names

print("\nFile sizes")
print(f"Source size GB: {src.stat().st_size / 1024**3:.3f}")
print(f"Output size GB: {out.stat().st_size / 1024**3:.3f}")

print("\nRow counts")
print("Source rows:", src_pf.metadata.num_rows)
print("Output rows:", out_pf.metadata.num_rows)

assert src_pf.metadata.num_rows == out_pf.metadata.num_rows, (
    "Row count mismatch: "
    f"source={src_pf.metadata.num_rows}, output={out_pf.metadata.num_rows}"
)

# ============================================================
# Schema / column checks
# ============================================================

print("\nSource columns:")
print(src_cols)

print("\nOutput columns:")
print(out_cols)

missing_original_cols = [c for c in src_cols if c not in out_cols]
assert not missing_original_cols, (
    "Output is missing original source columns: "
    f"{missing_original_cols}"
)

assert HASH_COL in out_cols, f"Output missing required column: {HASH_COL}"
assert SIMHASH_COL in out_cols, f"Output missing required column: {SIMHASH_COL}"

expected_cols = list(src_cols) + [HASH_COL, SIMHASH_COL]
assert out_cols == expected_cols, (
    "Output column order is not source columns + hash + simhash.\n"
    f"Expected: {expected_cols}\n"
    f"Actual:   {out_cols}"
)

print("\nSchema check: OK")
print("All original columns preserved.")
print("hash and simhash added.")
print("Column order is correct.")

# ============================================================
# Content and recomputation checks
# ============================================================

sample_size = min(SAMPLE_ROWS, src_pf.metadata.num_rows)

src_batch = next(src_pf.iter_batches(columns=[TEXT_COL], batch_size=sample_size))
out_batch = next(out_pf.iter_batches(columns=[TEXT_COL, HASH_COL, SIMHASH_COL], batch_size=sample_size))

src_df = src_batch.to_pandas(ignore_metadata=True)
out_df = out_batch.to_pandas(ignore_metadata=True)

# Check text unchanged for sampled rows.
text_same = (src_df[TEXT_COL] == out_df[TEXT_COL]).all()
assert text_same, "Sample text rows do not match between source and output."

# Recompute hash/simhash from text only.
out_df["recomputed_hash"] = out_df[TEXT_COL].map(stable_text_hash)
out_df["recomputed_simhash"] = out_df[TEXT_COL].map(simhash_64_fast).astype("uint64")

hash_match = (out_df[HASH_COL] == out_df["recomputed_hash"]).all()
simhash_match = (out_df[SIMHASH_COL] == out_df["recomputed_simhash"]).all()

assert hash_match, "Hash check failed: saved hash does not match recomputed text hash."
assert simhash_match, "SimHash check failed: saved simhash does not match recomputed text simhash."

print("\nSample content check: OK")
print(f"Checked first {sample_size} rows.")
print("Text unchanged:", text_same)
print("Hash recomputation matched:", hash_match)
print("SimHash recomputation matched:", simhash_match)

display(out_df[[TEXT_COL, HASH_COL, "recomputed_hash", SIMHASH_COL, "recomputed_simhash"]].head())

print("\nFINAL RESULT: PASS")
print(f"part_{PART_ID}.parquet is valid.")

Checking parquet pair
Source: /home/jovyan/data/AUTokens50/part_56.parquet
Output: /home/jovyan/notebook/outputs/AUTokens50_hash_simhash/part_56.parquet

File sizes
Source size GB: 1.088
Output size GB: 0.759

Row counts
Source rows: 202508
Output rows: 202508

Source columns:
['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count', 'index']

Output columns:
['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count', 'index', 'hash', 'simhash']

Schema check: OK
All original columns preserved.
hash and simhash added.
Column order is correct.

Sample content check: OK
Checked first 100 rows.
Text unchanged: True
Hash recomputation matched: True
SimHash recomputation matched: True


,text,hash,recomputed_hash,simhash,recomputed_simhash
0,Let me give you the context first…\nFor what s...,a76a443612c33c4d58c5325c8f779daad3128c8d,a76a443612c33c4d58c5325c8f779daad3128c8d,4198082839671602238,4198082839671602238
1,Thank you to our Guest Contributor Breanna How...,77c07ce14f0d07c6ab53e9bfa9e400b5897c1217,77c07ce14f0d07c6ab53e9bfa9e400b5897c1217,8791807371204357151,8791807371204357151
2,Ipswich City Council is continuing to monitor ...,d1adf7d19df39f928ef8ad288c7ebbe790183092,d1adf7d19df39f928ef8ad288c7ebbe790183092,17443148172498787934,17443148172498787934
3,The 5th Battalions have had a long and disting...,a0df5a78970981dfc5a77ec3e1bc044316e2c11e,a0df5a78970981dfc5a77ec3e1bc044316e2c11e,4186683612634806814,4186683612634806814
4,Thomas Isaac (Name Changed to protect his iden...,9b832ff3382b582cf7f49de5be103b667b7ab3d4,9b832ff3382b582cf7f49de5be103b667b7ab3d4,13477803157107767614,13477803157107767614



FINAL RESULT: PASS
part_56.parquet is valid.



---

## 2. Full output summary verification cell 的 Markdown

```markdown
## Summary Verification for All Processed Parquet Files

This cell performs a lightweight validation over the entire processed output directory.

Instead of reading all text content, it checks parquet metadata and schemas for every generated `part_*.parquet` file. This makes the check fast while still confirming the key structural requirements.

The cell verifies that:

- the expected number of output parquet files exists;
- each output file contains the required `hash` column;
- each output file contains the required `simhash` column;
- the original `index` column is preserved;
- row counts can be collected from all output files;
- the total number of output rows can be reported.

This cell should be run after all parquet files have been processed. It provides a final high-level confirmation that the output directory is structurally complete and ready for upload or downstream deduplication.

In [15]:
from pathlib import Path
import pyarrow.parquet as pq
import pandas as pd

OUTPUT_DIR = Path("/home/jovyan/notebook/outputs/AUTokens50_hash_simhash")

files = sorted(
    OUTPUT_DIR.glob("part_*.parquet"),
    key=lambda p: int(p.stem.split("_")[-1])
)

rows = []

for p in files:
    pf = pq.ParquetFile(p)
    cols = pf.schema.names
    rows.append({
        "file": p.name,
        "rows": pf.metadata.num_rows,
        "has_hash": "hash" in cols,
        "has_simhash": "simhash" in cols,
        "has_index": "index" in cols,
        "num_columns": len(cols),
    })

df = pd.DataFrame(rows)

display(df)

print("Output parquet files:", len(files))
print("Files missing hash:", (~df["has_hash"]).sum())
print("Files missing simhash:", (~df["has_simhash"]).sum())
print("Files missing index:", (~df["has_index"]).sum())
print("Total rows:", df["rows"].sum())

assert len(files) == 57
assert df["has_hash"].all()
assert df["has_simhash"].all()
assert df["has_index"].all()

print("FINAL RESULT: all output parquet files have hash, simhash, and preserved index.")

,file,rows,has_hash,has_simhash,has_index,num_columns
0,part_0.parquet,278106,True,True,True,12
1,part_1.parquet,374529,True,True,True,12
2,part_2.parquet,268518,True,True,True,12
3,part_3.parquet,342427,True,True,True,12
4,part_4.parquet,259925,True,True,True,12
5,part_5.parquet,288089,True,True,True,12
6,part_6.parquet,288631,True,True,True,12
7,part_7.parquet,295847,True,True,True,12
8,part_8.parquet,267184,True,True,True,12
9,part_9.parquet,277099,True,True,True,12


Output parquet files: 57
Files missing hash: 0
Files missing simhash: 0
Files missing index: 0
Total rows: 21087435
FINAL RESULT: all output parquet files have hash, simhash, and preserved index.
